# Temporary test development notebook

**Goal:** See if bias is being introduced artifically via our testing by randomizing various parameters.


#### 1. Imports and basic setup:

In [3]:
from __future__ import annotations
import dataclasses

import numpy as np
import matplotlib.pyplot as plt

from kinextract import (
    FitConfig, build_template_stack, gauss_hermite_model, compute_losvd_moments,
    bootstrap_losvd, load_emiles_templates, select_lambda_aic_local, whiten,
)
from kinextract.multi_template import fit_losvd

EMILES_GRID = "../../src/kinextract/template_data/emiles_muse_grid.npz"
EMILES_TEMPLATES = np.load(EMILES_GRID, allow_pickle=True)
WAVEMIN, STEP, N_PIX = 4750.0, 1.25, 3681  # MUSE pixel-sampling convention (the calibration's own domain)
WAVEFITMIN, WAVEFITMAX = 8400.0, 8800.0  # CaII triplet region, MUSE red arm

observed_wavelength = WAVEMIN + np.arange(N_PIX) * STEP
wave_fit = observed_wavelength[(observed_wavelength >= WAVEFITMIN) & (observed_wavelength <= WAVEFITMAX)]

#### 1. Main function to run a fit with one fixed parameter and the others randomized

In [4]:
def randomized_testing(
    true_v=None, true_sig=None, true_h3=None, true_h4=None, true_w=None, 
    metallicities=None, ages=None, SNR=None, plot_fit=False, gaussian_losvd=False, 
    bootstrap=False, n_bootstrap=None
):

    #### Build synthetic spectrum ####
    if metallicities is not None:
        # choose 4 random ages from the EMILES grid,
        age_input = np.random.choice(EMILES_TEMPLATES['ages'], size=4, replace=True)
        templates, template_wave, names, ages, metals = load_emiles_templates(EMILES_GRID, select=[(age_input[0], metallicities[0]), (age_input[1], metallicities[1]), 
                                                                                                   (age_input[2], metallicities[2]), (age_input[3], metallicities[3])],)
    elif ages is not None:
        metallicity_input = np.random.choice(EMILES_TEMPLATES['metals'], size=4, replace=True)
        templates, template_wave, names, ages, metals = load_emiles_templates(EMILES_GRID, select=[(ages[0], metallicity_input[0]), (ages[1], metallicity_input[1]), 
                                                                                                   (ages[2], metallicity_input[2]), (ages[3], metallicity_input[3])],)
    else:
        # choose 4 random ages and metallicities from the EMILES grid,
        age_input = np.random.choice(EMILES_TEMPLATES['ages'], size=4, replace=True)
        metallicity_input = np.random.choice(EMILES_TEMPLATES['metals'], size=4, replace=True)
        templates, template_wave, names, ages, metals = load_emiles_templates(EMILES_GRID, select=[(age_input[0], metallicity_input[0]), (age_input[1], metallicity_input[1]), 
                                                                                                   (age_input[2], metallicity_input[2]), (age_input[3], metallicity_input[3])],)
    TRUE_V = np.random.uniform(0.0, 200.0) if true_v is None else true_v
    TRUE_SIGMA = np.random.uniform(40.0, 400.0) if true_sig is None else true_sig
    if gaussian_losvd:
        TRUE_H3 = 0.0
        TRUE_H4 = 0.0
    else:
        TRUE_H3 = np.random.uniform(-0.1, 0.1) if true_h3 is None else true_h3
        TRUE_H4 = np.random.uniform(-0.1, 0.1) if true_h4 is None else true_h4
    TRUE_W = np.random.dirichlet(np.ones(len(templates))) if true_w is None else true_w
    SNR = np.random.uniform(20.0, 400.0) if SNR is None else SNR

    cfg.vmin=TRUE_V - 7 * TRUE_SIGMA 
    cfg.vmax=TRUE_V + 7 * TRUE_SIGMA
    v_grid = cfg.build_velocity_grid()
    b_truth = np.maximum(gauss_hermite_model(v_grid, amp=1.0, vel=TRUE_V, sig=TRUE_SIGMA, h3=TRUE_H3, h4=TRUE_H4), 0.0)
    b_truth /= b_truth.sum()

    A_stack = build_template_stack(wave_fit, template_wave, templates, v_grid)
    galaxy_noiseless = sum(TRUE_W[k] * (A_stack[k] @ b_truth) for k in range(len(templates)))
    design_true = np.stack([A_stack[k] @ b_truth for k in range(len(templates))], axis=1)
    corr = np.corrcoef(design_true.T)

    noise_level = np.median(galaxy_noiseless) / SNR
    errors = np.full(len(wave_fit), noise_level)
    rng = np.random.default_rng()
    galaxy = galaxy_noiseless + rng.normal(0.0, noise_level, len(wave_fit))
    A_stack_w, y_w, L = whiten(A_stack, galaxy, errors, rho=0.0)

    ### Fit the synthetic spectrum and extract LOSVD Moments ###
    result = fit_losvd(A_stack_w, y_w, cfg)
    moments = compute_losvd_moments(v_grid, result['b'], fit_h3h4=True)
    w_frac = result['w'] / result['w'].sum()
    b_truth_unit = b_truth / np.trapezoid(b_truth, v_grid)
    moments_truth = compute_losvd_moments(v_grid, b_truth_unit, fit_h3h4=True)
    bias = {k: moments[k] - moments_truth[k] for k in ['vherm', 'sherm', 'h3', 'h4']}

    A_eff_w = np.tensordot(result['w'], A_stack_w, axes=([0], [0]))
    model_w = A_eff_w @ result['b']
    b_truth_n = b_truth / np.trapezoid(np.maximum(b_truth, 0.0), v_grid)
    b_fit_n = result['b'] / np.trapezoid(np.maximum(result['b'], 0.0), v_grid)

    if bootstrap==True:
        n = n_bootstrap if n_bootstrap is not None else 30
        bootstrap_results = bootstrap_losvd(A_stack_w, y_w, cfg, n_boot=n, apply_bias_correction=True)
        b_bootstrap = np.array([res['b'] for res in bootstrap_results])
        moments_bootstrap = [compute_losvd_moments(v_grid, b_bootstrap[i], fit_h3h4=True) for i in range(n)]
        moments_bootstrap = {k: np.array([m[k] for m in moments_bootstrap]) for k in ['vherm', 'sherm', 'h3', 'h4']}
        bias_bootstrap = {k: moments_bootstrap[k] - moments_truth[k] for k in ['vherm', 'sherm', 'h3', 'h4']}
    else:
        bias_bootstrap = {k: None for k in ['vherm', 'sherm', 'h3', 'h4']}

    if plot_fit==True:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        ax = axes[0]
        ax.plot(wave_fit, galaxy, lw=1.0, color='black', alpha=0.7, label='galaxy')
        ax.plot(wave_fit, (L @ model_w), lw=1.6, color='tomato', label='model (LOSVD+weights)')
        ax.set_xlabel('wavelength (Angstrom)'); ax.set_ylabel('flux'); ax.legend(fontsize=8)
        ax.set_title('Spectral fit')

        ax = axes[1]
        ax.plot(v_grid, b_truth_n, color='black', lw=2.0, ls='--', label='true LOSVD')
        
        if bootstrap==True:
            ax.fill_between(v_grid, np.percentile(b_bootstrap, 16, axis=0), np.percentile(b_bootstrap, 84, axis=0), color='seagreen', alpha=0.3, label='68% CI (bootstrap)')
            ax.plot(v_grid, np.median(b_bootstrap, axis=0), color='seagreen', lw=1.5, ls=':', label='recovered')
        else:
            ax.plot(v_grid, b_fit_n, color='seagreen', lw=2.0, label='recovered')
        ax.set_xlabel('velocity (km/s)'); ax.set_ylabel(r'LOSVD (km/s)$^{-1}$'); ax.legend(fontsize=8)
        ax.set_xlim(TRUE_V - 4 * TRUE_SIGMA, TRUE_V + 4 * TRUE_SIGMA)
        ax.set_title('Recovered vs. true LOSVD')
        plt.tight_layout(); plt.show()

    results = {
        'true_v': TRUE_V,
        'true_sig': TRUE_SIGMA,
        'true_h3': TRUE_H3,
        'true_h4': TRUE_H4,
        'N_templates': len(templates),
        'true_w': TRUE_W,
        'metallicities': metals,
        'ages': ages,
        'SNR': SNR,
        'fit_v': moments['vherm'],
        'fit_sig': moments['sherm'],
        'fit_h3': moments['h3'],
        'fit_h4': moments['h4'],
        'bias_v': bias['vherm'],
        'bias_sig': bias['sherm'],
        'bias_h3': bias['h3'],
        'bias_h4': bias['h4'],
        'bias_v_bootstrap': bias_bootstrap['vherm'],
        'bias_sig_bootstrap': bias_bootstrap['sherm'],
        'bias_h3_bootstrap': bias_bootstrap['h3'],
        'bias_h4_bootstrap': bias_bootstrap['h4'],
    }
    return results, b_truth_n, b_fit_n, galaxy, (L @ model_w)

#### 2. Set up the kinextract configuration

In [5]:
cfg = FitConfig(
          vmin                    = -600.0,
          vmax                    = 600.0,
          bin_spacing_kms         = 8.0,
          min_bins                = 149,
          n_interior_knots        = None,
          knot_spacing_kms        = 95.0,
          min_interior_knots      = 16,
          degree                  = 3,
          continuum_knots         = None,
          lam_c                   = 2.0,
          ridge_c                 = 0.1,
          lam                     = 2.0,
          ridge_lam               = 1.0,
          lam_selection           = 'aic_local',
          lam_grid                = None,
          penalty_order           = 3,
          taper_sfac              = 1.8,
          lam_wing                = 0.0,
          taper_power             = 4.0,
          max_edge_amplitude_frac = 0.01,
          n_sigl0_iter            = 3,
          losvd_single_start      = False,
          fit_h3h4                = True,
      )

#### 2. Test function for a single fixed value of $v$

In [6]:
true_SNR = 100.0    # realistic per-bin S/N for a MUSE galaxy spaxel/bin
true_V = 40.0
true_Sigma = 120.0
true_H3 = 0.05
true_H4 = 0.02


# need to figure out why bootstrapping is apparently stalled?
v_test, b_truth_n, b_fit_n, spectrum, best_fit = randomized_testing(SNR=true_SNR, 
                                                                    true_v=true_V, 
                                                                    true_sig=true_Sigma, 
                                                                    true_h3=true_H3, 
                                                                    true_h4=true_H4,
                                                                    plot_fit=True, bootstrap=True, n_bootstrap=10)

TypeError: string indices must be integers, not 'str'

#### 2. Run a bias check for an array of input SNR values

In [5]:
true_SNR_array = np.array([20.0, 40.0, 60.0, 100.0, 150.0, 200.0, 300.0, 500.0, 800.0])

result_array = []
b_true_n_array = []
b_fit_n_array = []
spectrum_array = []
best_fit_array = []

for SNR in true_SNR_array:
    result, b_truth_n, b_fit_n, spectrum, best_fit = randomized_testing(SNR=SNR, plot_fit=False)
    result_array.append(result)
    b_true_n_array.append(b_truth_n)
    b_fit_n_array.append(b_fit_n)
    spectrum_array.append(spectrum)
    best_fit_array.append(best_fit)

Input true moments: v=76.64, sig=53.61, h3=-0.000, h4=0.084
GH fit true moments: v=76.63, sig=62.35, h3=-0.000, h4=0.089
Recovered moments: v=70.94, sig=69.12, h3=-0.000, h4=0.000
Input true moments: v=109.46, sig=272.01, h3=0.009, h4=0.047
GH fit true moments: v=113.65, sig=296.20, h3=0.011, h4=0.050
Recovered moments: v=101.55, sig=265.20, h3=0.002, h4=0.001
Input true moments: v=72.55, sig=394.94, h3=0.079, h4=-0.019
GH fit true moments: v=113.84, sig=387.50, h3=0.070, h4=-0.023
Recovered moments: v=73.35, sig=412.34, h3=0.021, h4=0.005
Input true moments: v=56.82, sig=118.78, h3=-0.028, h4=0.002
GH fit true moments: v=51.31, sig=118.01, h3=-0.028, h4=0.000
Recovered moments: v=52.93, sig=118.51, h3=-0.005, h4=0.000
Input true moments: v=46.67, sig=370.54, h3=-0.095, h4=0.094
GH fit true moments: v=-11.70, sig=433.40, h3=-0.117, h4=0.068
Recovered moments: v=2.26, sig=432.77, h3=-0.101, h4=0.071
Input true moments: v=48.39, sig=305.76, h3=-0.022, h4=0.043
GH fit true moments: v=36.8

In [1]:
# plot the recovered LOSVD moments as a function of SNR vs TRUE moments
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

for i, (ax, moment) in enumerate(zip(ax.flat, ['v', 'sig', 'h3', 'h4'])):
    ax.plot(true_SNR_array, [r[f'true_{moment}'] for r in result_array], 'o-', label='True')
    ax.plot(true_SNR_array, [r[f'fit_{moment}'] for r in result_array], 's-', label='Recovered')
    ax.set_xlabel('SNR')
    ax.set_ylabel(f'{moment}')
    ax.legend()
    ax.set_title(f'{moment} vs SNR')

NameError: name 'plt' is not defined